In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
from core.utils import load_predictions, add_confusion_categories
from core.utils import save_and_show


In [ ]:
# add_confusion_categories is imported from core.utils (see cell 0)


# **Load data**

In [ ]:
from config import (
    USE_DUMMY_DATA,
    FUSION_RUN_PATH,
    SELECTED_VARIABLES,
    VAF_DATA_PATH_BM, VAF_DATA_PATH_PB,
    SURVIVAL_DATA_PATH_BM, SURVIVAL_DATA_PATH_PB,
)

MODEL_PREDICTIONS_PATH_FUSION_BM = FUSION_RUN_PATH
MODEL_PREDICTIONS_PATH_FUSION_PB = FUSION_RUN_PATH


### FUSION

### Bone marrow

In [ ]:
df_vaf_bm = pd.read_excel(VAF_DATA_PATH_BM)
df_survival_bm = pd.read_excel(SURVIVAL_DATA_PATH_BM)

predicted_labels_bm = load_predictions(MODEL_PREDICTIONS_PATH_FUSION_BM, SELECTED_VARIABLES)
predicted_labels_bm = add_confusion_categories(predicted_labels_bm, SELECTED_VARIABLES, keep_original=True)

### Peripheral blood

In [ ]:
df_vaf_pb = pd.read_excel(VAF_DATA_PATH_PB)
df_survival_pb = pd.read_excel(SURVIVAL_DATA_PATH_PB)

#predicted_labels_pb = load_predictions(MODEL_PREDICTIONS_PATH_FUSION_PB, SELECTED_VARIABLES)
#predicted_labels_pb = add_confusion_categories(predicted_labels_pb, SELECTED_VARIABLES, keep_original=True)
#predicted_labels_pb.album_id = predicted_labels_pb.album_id.astype(str)

# **Association of predictions and VAF**

In [ ]:
from core.visualization.box_plot import plot_covariate_distributions
from core.visualization.scatter_plot import draw_scatter_plot

def run_all_experiments(df_vaf, predicted_labels, variables_to_plot=None):
    variables = [
        col[: -len("_predicted")]
        for col in predicted_labels.columns
        if col.endswith("_predicted")
    ]

    for variable_name in variables:
        if variables_to_plot is not None and variable_name not in variables_to_plot:
            continue

        if variable_name not in df_vaf.columns:
            print(f"{variable_name} not in VAF table")
            print("-----------------------------------------")
            continue

        # Build per-variable prediction slice
        pred_cols = [
            "album_id", "database_id",
            f"{variable_name}_predicted",
            f"{variable_name}_pred_prob",
        ]
        pred_slice = predicted_labels[[c for c in pred_cols if c in predicted_labels.columns]].copy()
        pred_slice = pred_slice.dropna(subset=f"{variable_name}_predicted")
        pred_slice = pred_slice.rename(columns={
            f"{variable_name}_predicted": "predicted",
            f"{variable_name}_pred_prob": "pred_prob",
        })
        
        df = df_vaf[["henkilotunnus", "album_id", "database_id", variable_name]].merge(
            pred_slice, on=["album_id", "database_id"], how="inner"
        )

        # drop samples with no VAF value
        df = df.dropna(subset=variable_name)

        # drop VAF outliers
        df = df[df[variable_name] <= 100]

        # analyse only truly positive samples (TP + FN)
        df = df[df["predicted"].isin(["TP", "FN"])]
        df["true_positive"] = df["predicted"] == "TP"

        if len(df) > 0:
            print(df["true_positive"].value_counts())

            try:
                print(variable_name)
                fig = plot_covariate_distributions(
                    df,
                    {variable_name: variable_name},
                    "true_positive",
                    fontsize=7,
                    label_mapping={True: "True positive", False: "False negative"},
                    xlabel="",
                    ylabel="VAF (%)",
                    num_columns=1,
                    fig_width=2,
                    fig_height=2,
                )
                save_and_show(fig, figure_type="main")

                draw_scatter_plot(
                    df=df,
                    x_variable="pred_prob",
                    y_variable=variable_name,
                    correlation_method="spearman",
                    show_trendline=False,
                    show_regression=True,
                    show_correlation=True,
                    title=variable_name,
                    x_label="Confidence",
                    y_label="VAF (%)",
                    fontsize=7,
                    show_grid=False,
                    figsize=(2, 2),
                )
            except:
                plt.clf()
        print("-----------------------------------------")

### Bone marrow

In [ ]:
df_cyto = df_survival_bm[df_survival_bm.cyto == True]
patients_to_include = df_cyto.henkilotunnus.to_list()
df_vaf_bm_cyto = df_vaf_bm[df_vaf_bm.henkilotunnus.isin(patients_to_include)]

# cytostasis treated patients only
print("CYTOSTASIS PATIENTS")
run_all_experiments(df_vaf_bm_cyto, predicted_labels_bm)

# all available patients
print("ALL PATIENTS")
run_all_experiments(df_vaf_bm, predicted_labels_bm)

### Peripheral blood

In [ ]:
run_all_experiments(df_vaf_pb, PREDICTION_RESULTS_DIR_PB)